In [ ]:
# Use the Forest Landscape Integrity Index (Grantham et al., 2020) to make a 
# mask for regional datasets, excluding low integrity land areas
# 
# Best to run this on a dedicated CPU node

In [1]:
import rioxarray
import xarray as xr
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape

In [2]:
# Open the original file using 'chunks' (enables lazy loading)
da = rioxarray.open_rasterio("/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ForestLandscapeIntegrityIndex/flii_SouthAmerica.tif", chunks={'x': 2048, 'y': 2048})

# Define bounding box in the CRS of data (usually Lat/Lon if WGS84 / EPSG:4326)
# Format: [minx, miny, maxx, maxy] -> [min_lon, min_lat, max_lon, max_lat]
bbox = [-82, -22, -42, 12]  

# Clip the dataset to just this bounding box
da_clipped = da.rio.clip_box(*bbox, crs="EPSG:4326")

# Compute threshold on the much smaller subset
threshold_value = 6000 # exclude low integrity land areas (index < 6)
mask_clipped = da_clipped > threshold_value

# Convert to uint8 (0 and 1) to save space
mask_clipped = mask_clipped.astype("uint8")
mask_clipped.name = "small_mask"

# Save the subset mask
mask_clipped.rio.to_raster("/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ForestLandscapeIntegrityIndex/flii_6_Amazon.tif")

In [3]:
# Make a shapefile from the GeoTIFF mask
mask_path = "/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ForestLandscapeIntegrityIndex/flii_6_Amazon.tif"

with rasterio.open(mask_path) as src:
    mask_image = src.read(1) # Read the mask layer
    
    # Extract the geometry shapes where the pixel value equals 1
    # mask_image == 1 filters out the background (0) pixels
    shape_generator = shapes(
        mask_image, 
        mask=(mask_image == 1), 
        transform=src.transform
    )
    
    # Convert the generator results into Shapely geometry objects
    polygons = []
    for geometry, value in shape_generator:
        polygons.append(shape(geometry))

# Convert the list of polygons into a GeoDataFrame
gdf = gpd.GeoDataFrame(geometry=polygons, crs=src.crs)

# Save as a Shapefile 
gdf.to_file("/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ForestLandscapeIntegrityIndex/flii_6_Amazon_mask_polygon.shp")